In [1]:
import poolparty as pp 
highlights = [
    pp.Highlighter(which='tags', style='green'),
    pp.Highlighter(region='cre', style='yellow'),
    pp.Highlighter(region='cre', which='lower', style='red'),
    pp.Highlighter(region='cre', which='gap', style='red'),
    pp.Highlighter(region='bc', style='cyan')
]

In [2]:
pp.init()
pp.set_highlights(highlights)

# 1. Create template Pool
pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA')\
    .print_library()

pool[0]: seq_length=51, num_states=1
state  seq
    0  TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA



In [3]:
pp.init()
pp.set_highlights(highlights)

# 1. Create template Pool
# 2. Randomly mutate cre in 5 different ways
pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA')\
    .mutagenize('cre', mutation_rate=0.1, mode='hybrid', num_hybrid_states=5, seq_name_prefix='mut_')\
    .print_library()

pool[1]: seq_length=51, num_states=5
state  name   seq
    0  mut_0  TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc/>AGATCGGA
    1  mut_1  TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc/>AGATCGGA
    2  mut_2  TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc/>AGATCGGA
    3  mut_3  TCCCGACT<cre>GGAAtGCGGGCAGTGAaatCACAGGgAA</cre>ATTACGG<bc/>AGATCGGA
    4  mut_4  TCCCGACT<cre>GGgAAGCGGGCtGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA



In [4]:
pp.init()
pp.set_highlights(highlights)

# 1. Create template Pool
# 2. Randomly mutate <cre>...</cre> region in 5 different ways
# 3. Repeat each mutant 3 times
# 4. Insert random barcodes at <bc/>
pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA')\
    .mutagenize('cre', mutation_rate=0.1, mode='hybrid', num_hybrid_states=5, seq_name_prefix='mut_')\
    .repeat_states(3, seq_name_prefix='v')\
    .insert_kmers('bc', length=5)\
    .print_library()

pool[3]: seq_length=5, num_states=15
state  name      seq
    0  mut_0.v0  TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc>tggcc</bc>AGATCGGA
    1  mut_0.v1  TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc>aaaat</bc>AGATCGGA
    2  mut_0.v2  TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc>gtggt</bc>AGATCGGA
    3  mut_1.v0  TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc>ggggt</bc>AGATCGGA
    4  mut_1.v1  TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc>ctgac</bc>AGATCGGA
    5  mut_1.v2  TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc>tgatg</bc>AGATCGGA
    6  mut_2.v0  TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc>taata</bc>AGATCGGA
    7  mut_2.v1  TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc>gaccc</bc>AGATCGGA
    8  mut_2.v2  TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc>caaaa</bc>AGATCGGA
    9  mut_3.v0  TCCCGACT<cre>GGAAtGCGGGCAGTGAaatCACAGGgAA</cre>ATTACGG<bc>gggcg</bc>AGATCGGA
  

In [5]:
pp.init()
pp.set_highlights(highlights)

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA')

wildtype_pool = template_pool\
    .repeat_states(3, seq_name_prefix='wt.v')\
    .named('wildtype_pool')

mutated_pool = template_pool\
    .mutagenize('cre', mutation_rate=0.1, mode='hybrid', num_hybrid_states=5, seq_name_prefix='mut_')\
    .repeat_states(3, seq_name_prefix='v')\
    .named('mutated_pool')

deletion_pool = template_pool\
    .deletion_scan('cre', deletion_length=5, positions=slice(None, None, 5), mode='sequential', seq_name_prefix='del_')\
    .repeat_states(3, seq_name_prefix='v')\
    .named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        mode='sequential', 
                        op_iter_order=-1,
                        seq_name_prefix='site_')
    
insertion_pool = template_pool\
    .insertion_scan('cre', ins_pool=sites_pool, positions=slice(None,None,5), replace=True, mode='sequential', seq_name_prefix='ins_')

combo_pool = pp.stack([wildtype_pool, mutated_pool, deletion_pool, insertion_pool])\
    .insert_kmers('bc', length=5)\
    .named('combo_pool')\
    .print_library()

combo_pool: seq_length=5, num_states=43
state  name          seq
    0  wt.v0         TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc>tggcc</bc>AGATCGGA
    1  wt.v1         TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc>aaaat</bc>AGATCGGA
    2  wt.v2         TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc>gtggt</bc>AGATCGGA
    3  mut_0.v0      TCCCGACT<cre>GGAAAGCGcGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc>ggggt</bc>AGATCGGA
    4  mut_0.v1      TCCCGACT<cre>GGAAAGCGcGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc>ctgac</bc>AGATCGGA
    5  mut_0.v2      TCCCGACT<cre>GGAAAGCGcGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc>tgatg</bc>AGATCGGA
    6  mut_1.v0      TCCCGACT<cre>GGAAAGCGaGCAGTGAGCcCACcGGAAA</cre>ATTACGG<bc>taata</bc>AGATCGGA
    7  mut_1.v1      TCCCGACT<cre>GGAAAGCGaGCAGTGAGCcCACcGGAAA</cre>ATTACGG<bc>gaccc</bc>AGATCGGA
    8  mut_1.v2      TCCCGACT<cre>GGAAAGCGaGCAGTGAGCcCACcGGAAA</cre>ATTACGG<bc>caaaa</bc>AGATCGGA
    9  mut_2.v0      TCCCGACT<cre>GGAAAGCGGGCAGTGAGCA

In [6]:
df = combo_pool.clear_annotation()\
    .generate_library(report_design_cards=False)
df.to_excel('~/Desktop/library.xlsx', index=False)

In [7]:
combo_pool.print_dag()

combo_pool (pool, n=43)
└── op[12]:get_kmers [mode=random, n=1]
    └── pool[11] (pool, n=43)
        └── op[11]:stack [mode=fixed, n=1]
            ├── wildtype_pool (pool, n=3)
            │   └── op[1]:repeat [mode=sequential, n=3]
            │       └── pool[0] (pool, n=1)
            │           └── op[0]:from_seq [mode=fixed, n=1]
            ├── mutated_pool (pool, n=15)
            │   └── op[3]:repeat [mode=sequential, n=3]
            │       └── pool[2] (pool, n=5)
            │           └── op[2]:mutagenize [mode=hybrid, n=5]
            │               └── pool[0] (pool, n=1)
            │                   └── op[0]:from_seq [mode=fixed, n=1]
            ├── deletion_pool (pool, n=15)
            │   └── op[6]:repeat [mode=sequential, n=3]
            │       └── pool[5] (pool, n=5)
            │           └── op[5]:deletion_scan(from_seq) [mode=fixed, n=1]
            │               └── pool[4] (pool, n=5)
            │                   └── op[4]:deletion_scan(marker

Pool(id=12, name='combo_pool', op='op[12]:get_kmers', num_states=43)

In [8]:
combo_pool.counter.print_dag()

combo_pool.state (counter, io=-1, n=43)
└── [op=Product]
    ├── pool[11].state (counter, io=-1, n=43)
    │   └── [op=Stack]
    │       ├── wildtype_pool.state (counter, io=0, n=3)
    │       │   └── [op=Product]
    │       │       ├── op[1]:repeat.state (counter, io=0, n=3)
    │       │       └── pool[0].state (counter, io=0, n=1)
    │       │           └── [op=Passthrough]
    │       │               └── op[0]:from_seq.state (counter, io=0, n=1)
    │       ├── mutated_pool.state (counter, io=0, n=15)
    │       │   └── [op=Product]
    │       │       ├── op[3]:repeat.state (counter, io=0, n=3)
    │       │       ├── op[2]:mutagenize.state (counter, io=0, n=5)
    │       │       └── pool[0].state (counter, io=0, n=1)
    │       │           └── [op=Passthrough]
    │       │               └── op[0]:from_seq.state (counter, io=0, n=1)
    │       ├── deletion_pool.state (counter, io=0, n=15)
    │       │   └── [op=Product]
    │       │       ├── op[6]:repeat.state (counter

In [9]:
counter_df = combo_pool.counter.get_iteration_df()
counter_df.to_excel('~/Desktop/counter_states.xlsx')

In [10]:
import statecounter as sc
A = sc.Counter(3, name='A')
B = sc.Counter(4, name='B')
C = sc.product([A,B], name='C')
C.get_iteration_df().T


C,0,1,2,3,4,5,6,7,8,9,10,11
A,0,1,2,0,1,2,0,1,2,0,1,2
B,0,0,0,1,1,1,2,2,2,3,3,3


In [11]:
import statecounter as sc
A = sc.Counter(3, name='A')
B = sc.Counter(4, name='B')
C = sc.stack([A,B], name='C')
C.get_iteration_df().T

C,0,1,2,3,4,5,6
A,0,1,2,None,None,None,None
B,None,None,None,0,1,2,3


In [12]:
import statecounter as sc
A = sc.Counter(2,name='A')
C = sc.repeat(A,3,name='C')
C.get_iteration_df().T

C,0,1,2,3,4,5
A,0,1,0,1,0,1


In [13]:
import statecounter as sc
A = sc.Counter(12,name='A')
C = sc.slice(A,None,None,3,name='C')
C.get_iteration_df().T

C,0,1,2,3
A,0,3,6,9
